In [1]:
!pip install -q transformers datasets

In [2]:
import random, re
from datasets import load_dataset
from transformers import pipeline

try:
    data = load_dataset("rotten_tomatoes", split="validation")
except Exception:
    data = load_dataset("cornell-movie-review-data/rotten_tomatoes", split="validation")

random.seed(42)
texts = [data[i]["text"] for i in random.sample(range(len(data)), 100)]
clf = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
predict = lambda t: clf(t, batch_size=16, truncation=True)
orig = predict(texts)

NEUTRAL = "I watched it on Tuesday."
def suffix(t): return t + " " + NEUTRAL
def prefix(t): return NEUTRAL + " " + t
def swap(t):
    new = re.sub(r"\bmovie\b", "film", t, flags=re.I)
    if new == t:
        new = re.sub(r"\bfilm\b", "movie", t, flags=re.I)
    return None if new == t else new

for name, f in [("neutral sentence at end", suffix),
                ("neutral sentence at start", prefix),
                ("movie/film swap", swap)]:
    pairs = [(t, o, f(t)) for t, o in zip(texts, orig) if f(t) is not None]
    new = predict([p[2] for p in pairs])
    flips = sum(o["label"] != n["label"] for (_, o, _), n in zip(pairs, new))
    print(f"{name}: label changed in {flips} of {len(pairs)} reviews")

README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

neutral sentence at end: label changed in 1 of 100 reviews
neutral sentence at start: label changed in 1 of 100 reviews
movie/film swap: label changed in 1 of 27 reviews


In [3]:
for name, f in [("end", suffix), ("start", prefix), ("movie/film swap", swap)]:
    pairs = [(t, o, f(t)) for t, o in zip(texts, orig) if f(t) is not None]
    new = predict([p[2] for p in pairs])
    for (t, o, c), n in zip(pairs, new):
        if o["label"] != n["label"]:
            print(f"[{name}] before: {o['label']} {o['score']:.2f} | after: {n['label']} {n['score']:.2f}")
            print("  ", t)

[end] before: NEGATIVE 0.99 | after: POSITIVE 0.78
   a movie to forget
[start] before: NEGATIVE 0.99 | after: POSITIVE 0.64
   a movie to forget
[movie/film swap] before: POSITIVE 0.79 | after: NEGATIVE 0.80
   a man leaving the screening said the film was better than saving private ryan . he may have meant the internet short saving ryan's privates . but windtalkers doesn't beat that one , either .
